In [1]:
# Add parent directory to path so we can import scripts module
import sys
import os

# Get the workspace root (parent of 'Eksamens notebooks')
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
workspace_root = os.path.dirname(notebook_dir)

if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

# 3D Drone Dynamics Analysis

Dette notebook demonstrerer beregning af drone dynamik i 3D ved hjælp af Newton-Euler ligninger.

Vi vil beregne:
- Rotationsmatrice fra lokal til global koordinatsystem
- Global inertitensor
- Total kraft på dronen
- Totalt moment (torque)
- Lineær acceleration
- Vinkelacceleration

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import importlib

# Import the 3D dynamics module
import scripts
importlib.reload(scripts)

from scripts import l_print
importlib.reload(l_print)
from scripts.l_print import lPrint

import scripts.drone_dynamics as drone_dynamics
importlib.reload(drone_dynamics)
from scripts.drone_dynamics import calculate_drone_dynamics

## Definer Drone Parametre

Vi definerer nu alle parametre for dronen:

In [3]:
# Euler vinkler (roll, pitch, yaw) i grader
roll_deg = 30   # Rotation omkring x-aksen
pitch_deg = 5   # Rotation omkring y-aksen
yaw_deg = 3     # Rotation omkring z-aksen

# Konverter til radianer
roll = np.deg2rad(roll_deg)
pitch = np.deg2rad(pitch_deg)
yaw = np.deg2rad(yaw_deg)

# Rotorkræfter i lokal koordinatsystem [Fx, Fy, Fz] (N)
# Antaget at rotorerne kun genererer kraft i z-retning
f_1 = np.array([[0], [0], [10]])   # Rotor 1 (foran)
f_2 = np.array([[0], [0], [25]])   # Rotor 2 (højre)
f_3 = np.array([[0], [0], [10]])   # Rotor 3 (bag)
f_4 = np.array([[0], [0], [20]])   # Rotor 4 (venstre)

# Dronens masse (kg)
drone_mass = 4

# Afstand fra massecentrum til hver rotor (m)
L = 0.4

# Lokal inertitensor (symmetrisk diagonal for symmetrisk drone)
I_local = np.array([[3, 0, 0],
                    [0, 3, 0],
                    [0, 0, 1]])

# Vinkelhastighed i globalt koordinatsystem [wx, wy, wz] (rad/s)
omega = np.array([0.1, 0.5, 0.1])

# Payload kraft i globalt koordinatsystem [Fx, Fy, Fz] (N) - valgfri
# Sæt til None hvis ingen payload
payload_f = None  # np.array([[6], [-3], [-30]]) for at inkludere payload

print("Parametre defineret!")
print(f"Euler angles: roll={roll_deg}°, pitch={pitch_deg}°, yaw={yaw_deg}°")
print(f"Drone masse: {drone_mass} kg")
print(f"Rotor afstand: {L} m")

Parametre defineret!
Euler angles: roll=30°, pitch=5°, yaw=3°
Drone masse: 4 kg
Rotor afstand: 0.4 m


## Beregn Drone Dynamik

Nu kalder vi funktionen der beregner alle dynamiske størrelser:

In [4]:
# Kald funktionen for at beregne dynamikken
results = calculate_drone_dynamics(
    roll=roll,
    pitch=pitch,
    yaw=yaw,
    f1=f_1,
    f2=f_2,
    f3=f_3,
    f4=f_4,
    drone_mass=drone_mass,
    L=L,
    I_local=I_local,
    omega=omega,
    payload_f=payload_f,
    XYZ=True,  # Brug X-Y-Z rotationskonvention
    useLPrint=True  # Brug formateret output
)

**Rotation omkring x-aksen (roll):**: $$R_x=\begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos(\phi) & -\sin(\phi) \\ 0 & \sin(\phi) & \cos(\phi) \end{bmatrix}$$

**Rotation omkring y-aksen (pitch):**: $$R_y=\begin{bmatrix} \cos(\theta) & 0 & \sin(\theta) \\ 0 & 1 & 0 \\ -\sin(\theta) & 0 & \cos(\theta) \end{bmatrix}$$

**Rotation omkring z-aksen (yaw):**: $$R_z=\begin{bmatrix} \cos(\psi) & -\sin(\psi) & 0 \\ \sin(\psi) & \cos(\psi) & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

**Total rotationsmatrice (X-Y-Z konvention):**: $$R=R_x \cdot R_y \cdot R_z$$

**Rotationsmatrice (numerisk):**: $$R=\begin{bmatrix}0.9948 & -0.0521 & 0.0872 \\ 0.0888 & 0.8626 & -0.4981 \\ -0.0492 & 0.5033 & 0.8627\end{bmatrix}$$

...

**Moment i globalt koordinatsystem:**: $$\tau_{global}=\begin{bmatrix}1.9897 & 0.1777 & -0.0984\end{bmatrix}$$

**Newtons 2. lov for lineær bevægelse:**: $$a=\frac{F_{total}}{m}$$

**Lineær acceleration (numerisk):**: $$a=\begin{bmatrix}1.4163 \\ -8.0941 \\ 4.1994\end{bmatrix}$$

**Eulers rotationsligning:**: $$\dot{\omega}=I^{-1} \cdot (\tau - \omega \times (I \cdot \omega))$$

**Vinkelacceleration (numerisk):**: $$\dot{\omega}=\begin{bmatrix}0.6138 & 0.0672 & -0.0232\end{bmatrix}$$

## Resultater

Nu kan vi udtrække de beregnede værdier:

In [5]:
# Udtræk resultater
R = results['rotation_matrix']
I_global = results['global_inertia']
F_total = results['total_force']
tau = results['total_torque']
a = results['linear_acceleration']
alpha = results['angular_acceleration']

print("\n" + "="*50)
print("SAMMENFATNING AF RESULTATER")
print("="*50)
print("\nRotationsmatrice R:")
print(R)
print("\nGlobal inertitensor I_global:")
print(I_global)
print("\nTotal kraft F_total [N]:")
print(F_total)
print("\nTotalt moment τ [Nm]:")
print(tau)
print("\nLineær acceleration a [m/s²]:")
print(a)
print("\nVinkelacceleration α [rad/s²]:")
print(alpha)
print("="*50)


SAMMENFATNING AF RESULTATER

Rotationsmatrice R:
[[ 0.99482945 -0.0521368   0.08715574]
 [ 0.08884242  0.86255786 -0.49809735]
 [-0.04920767  0.50326504  0.86272992]]

Global inertitensor I_global:
[[ 2.98480775  0.08682409 -0.15038373]
 [ 0.08682409  2.50379806  0.85944697]
 [-0.15038373  0.85944697  1.51139419]]

Total kraft F_total [N]:
[[  5.66512328]
 [-32.37632769]
 [ 16.79744452]]

Totalt moment τ [Nm]:
[ 1.9896589   0.17768483 -0.09841534]

Lineær acceleration a [m/s²]:
[[ 1.41628082]
 [-8.09408192]
 [ 4.19936113]]

Vinkelacceleration α [rad/s²]:
[ 0.61379975  0.06719395 -0.02321358]
